# PotatoDoc: Production Validation Pipeline**Complete 20-stage audit, training, evaluation, and production-readiness pipeline**| Phase | Stages | Purpose ||-------|--------|---------|| 1 - Trust the Data | Audit, Duplicates, Leakage, Split | Ensure data quality || 2 - Build the Model | Augmentation, Training | Train all 4 backbones || 3 - Find the Real Problem | IPD vs PLD, Error Analysis, Grad-CAM | Understand domain gap || 4 - Production Robustness | Corruptions, OOD, Calibration | Real-world readiness || 5 - Final Decision | Comparison, Ensemble, Gate | GREEN/YELLOW/RED verdict |**Dataset:** Irish Potato Dataset (IPD) + Potato Leaf Disease Dataset (PLD)**Models:** ConvNeXt-Tiny v1/v2, EfficientNetV2-B3, Swin-Tiny + Ensemble**Primary Metric:** Macro-F1

In [ ]:
# ============================================================# CELL 1: Setup & Installation# ============================================================import subprocess, sysdef install(pkg):    try:        __import__(pkg)    except ImportError:        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])for p in ["timm", "albumentations", "imagehash", "scikit-learn", "pandas",          "matplotlib", "seaborn", "opencv-python-headless", "scipy"]:    install(p)print("All dependencies installed.")

In [ ]:
# ============================================================# CELL 2: Imports & Configuration# ============================================================import os, json, time, random, hashlib, csvfrom pathlib import Pathfrom collections import Counter, defaultdictimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom PIL import Imageimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import DataLoader, Datasetfrom torchvision import transformsfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    classification_report, f1_score, confusion_matrix,    balanced_accuracy_score)from sklearn.utils.class_weight import compute_class_weightimport timmfrom timm.data import Mixupfrom timm.loss import SoftTargetCrossEntropy# ============================================================# CONFIG# ============================================================SEED = 42DATA_DIR = Path("/content/dataset")RESULTS_DIR = Path("/content/results")RESULTS_DIR.mkdir(parents=True, exist_ok=True)IPD_CLASSES = ["earlyblt", "healthy", "lateblt"]IPD_CLASS_NAMES = ["Early Blight", "Healthy", "Late Blight"]NUM_CLASSES = 3PLD_MAP = {"Fungi": 0, "Healthy": 1, "Phytophora": 2}PLD_CLEAN_SUBSET = {"Healthy", "Phytophora"}MODELS_CONFIG = [    {"name": "efficientnetv2_b3", "timm_name": "tf_efficientnetv2_b3", "img_size": 300, "batch_size": 32},    {"name": "convnext_tiny_v1", "timm_name": "convnext_tiny.fb_in22k", "img_size": 224, "batch_size": 64},    {"name": "convnext_tiny_v2", "timm_name": "convnext_tiny.fb_in22k", "img_size": 224, "batch_size": 64},    {"name": "swin_tiny", "timm_name": "swin_tiny_patch4_window7_224.ms_in22k", "img_size": 224, "batch_size": 64},]def set_seed(seed):    random.seed(seed); np.random.seed(seed)    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = Trueset_seed(SEED)if torch.cuda.is_available():    device = torch.device("cuda")    print(f"GPU: {torch.cuda.get_device_name(0)}")else:    device = torch.device("cpu")    print("CPU mode")IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD = [0.229, 0.224, 0.225]print(f"PyTorch {torch.__version__} | timm {timm.__version__}")

In [ ]:
# ============================================================# CELL 3: Mount Drive & Locate Data# ============================================================try:    from google.colab import drive    drive.mount('/content/drive')    DATA_DIR = Path("/content/drive/MyDrive/dataset")    print(f"Mounted Drive. DATA_DIR = {DATA_DIR}")except ImportError:    DATA_DIR = Path(r"C:\Users\shadb\Downloads\dataset")    print(f"Local mode. DATA_DIR = {DATA_DIR}")RESULTS_DIR.mkdir(parents=True, exist_ok=True)ipd_ok = all((DATA_DIR / c / c).exists() for c in IPD_CLASSES)print(f"IPD structure OK: {ipd_ok}")for c in IPD_CLASSES:    n = len(list((DATA_DIR / c / c).glob("*.*")))    print(f"  {c}/{c}/ : {n} images")pld_root = DATA_DIR / "PLD" / "Potato Leaf Disease Dataset in Uncontrolled Environment"print(f"\nPLD exists: {pld_root.exists()}")if pld_root.exists():    for d in sorted(pld_root.iterdir()):        if d.is_dir():            n = len(list(d.glob("*.*")))            mapped = PLD_MAP.get(d.name, "IGNORED")            print(f"  {d.name}: {n} images -> {mapped}")

---## Phase 1: Dataset Audit

In [ ]:
# ============================================================# CELL 4: Dataset Audit — Class Distribution & Corruption Check# ============================================================def scan_ipd():    records = []    for cls_idx, cls_name in enumerate(IPD_CLASSES):        cls_dir = DATA_DIR / cls_name / cls_name        for f in sorted(cls_dir.iterdir()):            if f.suffix.lower() in (".jpg", ".jpeg", ".png"):                records.append({"path": str(f), "class": cls_name, "class_idx": cls_idx})    return pd.DataFrame(records)print("Scanning IPD...")t0 = time.time()df_ipd = scan_ipd()print(f"  Scanned {len(df_ipd)} images in {time.time()-t0:.1f}s")print("\n=== IPD Class Distribution ===")dist = df_ipd["class"].value_counts()for cls, cnt in dist.items():    print(f"  {cls:15s}: {cnt:6d} ({cnt/len(df_ipd)*100:.1f}%)")print(f"  Imbalance ratio: {dist.max()/dist.min():.2f}:1")print("\n=== Corruption Check ===")corrupt = []t0 = time.time()for _, row in df_ipd.iterrows():    try:        with Image.open(row["path"]) as img:            img.verify()    except Exception as e:        corrupt.append({"path": row["path"], "error": str(e)})print(f"  Checked {len(df_ipd)} images in {time.time()-t0:.1f}s")print(f"  Corrupted: {len(corrupt)}")if corrupt:    for c in corrupt[:5]:        print(f"    {c['path']}: {c['error']}")df_ipd["corrupted"] = df_ipd["path"].isin([c["path"] for c in corrupt])

In [ ]:
# ============================================================# CELL 5: SHA256 Exact Duplicate Detection# ============================================================print("Computing SHA256 hashes...")t0 = time.time()hashes = []for p in df_ipd["path"]:    h = hashlib.sha256()    with open(p, "rb") as f:        for chunk in iter(lambda: f.read(8192), b""):            h.update(chunk)    hashes.append(h.hexdigest())df_ipd["sha256"] = hashesprint(f"  Hashed {len(df_ipd)} images in {time.time()-t0:.1f}s")dup_groups = df_ipd.groupby("sha256").filter(lambda x: len(x) > 1)n_dup = dup_groups["sha256"].nunique() if len(dup_groups) > 0 else 0print(f"\n=== Exact Duplicates ===")print(f"  Duplicate groups: {n_dup}")print(f"  Total duplicate images: {len(dup_groups)}")if n_dup > 0:    for h, grp in list(dup_groups.groupby("sha256"))[:5]:        print(f"    Hash {h[:16]}...: {len(grp)} copies")        for _, r in grp.head(2).iterrows():            print(f"      {r['path']}")# Cross-class duplicatescross = 0for h, grp in dup_groups.groupby("sha256"):    if grp["class"].nunique() > 1:        cross += 1        print(f"  CROSS-CLASS DUP: {grp['class'].tolist()}")print(f"Cross-class exact duplicates: {cross}")

In [ ]:
# ============================================================# CELL 6: Perceptual Hash Near-Duplicate Detection# ============================================================import imagehashprint("Computing perceptual hashes (pHash)...")t0 = time.time()phashes = []for p in df_ipd["path"]:    try:        img = Image.open(p)        ph = imagehash.phash(img)        phashes.append(str(ph))    except Exception:        phashes.append(None)df_ipd["phash"] = phashesprint(f"  Done in {time.time()-t0:.1f}s")from imagehash import hex_to_hashvalid = df_ipd[df_ipd["phash"].notna()].reset_index(drop=True)phash_objs = [hex_to_hash(h) for h in valid["phash"]]near_dups = []n = len(valid)print(f"Checking {n} images for near-duplicates (distance <= 5)...")for i in range(n):    for j in range(i + 1, min(i + 300, n)):        dist = phash_objs[i] - phash_objs[j]        if dist <= 5:            near_dups.append({                "path_a": valid.iloc[i]["path"], "path_b": valid.iloc[j]["path"],                "class_a": valid.iloc[i]["class"], "class_b": valid.iloc[j]["class"],                "distance": dist            })print(f"Near-duplicate pairs: {len(near_dups)}")cross_near = [d for d in near_dups if d["class_a"] != d["class_b"]]print(f"Cross-class near-duplicates: {len(cross_near)}")for d in cross_near[:5]:    print(f"  {d['class_a']} <-> {d['class_b']} (dist={d['distance']})")

In [ ]:
# ============================================================# CELL 7: Leakage-Safe Stratified Split# ============================================================paths = df_ipd["path"].tolist()labels = df_ipd["class_idx"].tolist()X_train, X_temp, y_train, y_temp = train_test_split(    paths, labels, test_size=0.3, random_state=SEED, stratify=labels)X_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")assert len(set(X_train) & set(X_val)) == 0assert len(set(X_train) & set(X_test)) == 0assert len(set(X_val) & set(X_test)) == 0print("No overlap between splits - OK")for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:    counts = Counter(y)    print(f"\n{name} ({len(y)}):")    for i, c in enumerate(IPD_CLASSES):        print(f"  {c:15s}: {counts.get(i,0):6d} ({counts.get(i,0)/len(y)*100:.1f}%)")json.dump({"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val,           "X_test": X_test, "y_test": y_test}, open(RESULTS_DIR / "split_cache.json", "w"))print("\nSplit saved to results/split_cache.json")

In [ ]:
# ============================================================# CELL 8: Load PLD External Dataset# ============================================================def collect_pld():    root = DATA_DIR / "PLD" / "Potato Leaf Disease Dataset in Uncontrolled Environment"    paths, labels, groups = [], [], []    for folder, lab in PLD_MAP.items():        d = root / folder        if not d.exists():            print(f"  WARNING: {d} not found")            continue        fs = sorted(str(p) for p in d.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))        paths += fs; labels += [lab] * len(fs); groups += [folder] * len(fs)        print(f"  {folder} -> {IPD_CLASS_NAMES[lab]}: {len(fs)} images")    return paths, np.array(labels), groupsprint("=== PLD Dataset ===")pld_paths, pld_labels, pld_groups = collect_pld()print(f"Total PLD mapped: {len(pld_paths)}")clean_mask = np.array([g in PLD_CLEAN_SUBSET for g in pld_groups])print(f"Clean subset: {clean_mask.sum()} images")

---## Phase 2: Augmentation & Training

In [ ]:
# ============================================================# CELL 9: Augmentation Pipelines# ============================================================try:    import albumentations as A    from albumentations.pytorch import ToTensorV2    HAS_ALB = Trueexcept ImportError:    HAS_ALB = Falsedef get_transforms(img_size, is_train=True, strong=False):    norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)    if HAS_ALB:        if is_train:            tfs = [                A.RandomResizedCrop(img_size, img_size, scale=(0.5 if strong else 0.7, 1.0)),                A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2),            ]            if strong:                tfs += [A.OneOf([A.RandomBrightnessContrast(0.3, 0.3, p=1),                                  A.HueSaturationValue(20, 30, 20, p=1)], p=0.8),                        A.OneOf([A.GaussianBlur(3, p=1), A.GaussNoise(10, 50, p=1)], p=0.3),                        A.Rotate(limit=15, p=0.5)]            else:                tfs += [A.ColorJitter(0.2, 0.2, 0.2, 0.1, p=0.5), A.Rotate(limit=15, p=0.5)]            tfs += [A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()]            return A.Compose(tfs)        return A.Compose([A.Resize(int(img_size*1.14), int(img_size*1.14)),                          A.CenterCrop(img_size, img_size),                          A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])    # Fallback    if is_train:        tfs = [transforms.RandomResizedCrop(img_size, scale=(0.5 if strong else 0.7, 1.0)),               transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(p=0.2)]        if strong:            tfs += [transforms.RandAugment(num_ops=2, magnitude=9), transforms.RandomGrayscale(p=0.1)]        else:            tfs += [transforms.ColorJitter(0.2, 0.2, 0.2, 0.1), transforms.RandomRotation(15)]        tfs += [transforms.ToTensor(), norm]        return transforms.Compose(tfs)    return transforms.Compose([transforms.Resize(int(img_size*1.14)), transforms.CenterCrop(img_size),                               transforms.ToTensor(), norm])print("Augmentation ready.")

In [ ]:
# ============================================================# CELL 10: Dataset & Model Factory# ============================================================class PotatoDataset(Dataset):    def __init__(self, paths, labels, transform=None):        self.paths, self.labels, self.transform = paths, labels, transform    def __len__(self): return len(self.paths)    def __getitem__(self, idx):        try:            img = Image.open(self.paths[idx]).convert("RGB")            if self.transform:                if HAS_ALB and hasattr(self.transform, '__call__') and not isinstance(self.transform, transforms.Compose):                    img = self.transform(image=np.array(img))["image"]                else:                    img = self.transform(img)            return img, self.labels[idx]        except Exception:            return self.__getitem__(random.randint(0, len(self)-1))def make_loaders(Xtr, ytr, Xva, yva, img_size, batch_size, strong=False):    tr = PotatoDataset(Xtr, ytr, get_transforms(img_size, True, strong))    va = PotatoDataset(Xva, yva, get_transforms(img_size, False))    nw = min(4, os.cpu_count() or 1)    kw = dict(num_workers=nw, pin_memory=torch.cuda.is_available())    return (DataLoader(tr, batch_size, shuffle=True, **kw),            DataLoader(va, batch_size*2, shuffle=False, **kw))def build_model(timm_name, num_classes=NUM_CLASSES, drop_path=0.0):    model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes,                              drop_path_rate=drop_path)    return modeldef cpu_sd(sd): return {k: v.detach().cpu() for k, v in sd.items()}def save_ckpt(path, payload):    tmp = str(path) + ".tmp"; torch.save(payload, tmp); os.replace(tmp, path)print("Dataset & model factory ready.")

In [ ]:
# ============================================================# CELL 11: Training Loop# ============================================================def train_one_epoch(model, loader, crit, opt, scaler, mix_fn, device):    model.train(); tl, correct, total = 0.0, 0, 0    for x, y in loader:        x, y = x.to(device), y.to(device)        yin = y        if mix_fn is not None:            x, yin = mix_fn(x, y)        opt.zero_grad(set_to_none=True)        with torch.amp.autocast(device.type if device.type == "cuda" else "cpu"):            out = model(x); loss = crit(out, yin)        scaler.scale(loss).backward(); scaler.unscale_(opt)        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        scaler.step(opt); scaler.update()        tl += loss.item()*x.size(0); correct += (out.argmax(1)==y).sum().item(); total += x.size(0)    return tl/total, correct/total@torch.no_grad()def evaluate(model, loader, device):    model.eval(); P, L, tl, tot = [], [], 0.0, 0    ce = nn.CrossEntropyLoss()    for x, y in loader:        x, y = x.to(device), y.to(device)        o = model(x); tl += ce(o,y).item()*x.size(0); tot += x.size(0)        P.append(o.argmax(1).cpu().numpy()); L.append(y.cpu().numpy())    P, L = np.concatenate(P), np.concatenate(L)    return {"loss": tl/tot, "accuracy": float((P==L).mean()),            "macro_f1": float(f1_score(L, P, average="macro")),            "predictions": P, "labels": L}def train_model(name, timm_name, img_size, bs, Xtr, ytr, Xva, yva, device,                strong=True, label_smooth=0.1, wd=0.05, dp=0.1, patience=10):    ckpt_path = RESULTS_DIR / f"{name}_best.pth"    if ckpt_path.exists():        print(f"[SKIP] {name} checkpoint exists"); return torch.load(ckpt_path, map_location="cpu", weights_only=False)    set_seed(SEED)    tr_loader, va_loader = make_loaders(Xtr, ytr, Xva, yva, img_size, bs, strong)    model = build_model(timm_name, drop_path=dp).to(device)    cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=np.array(ytr))    cw = torch.FloatTensor(cw).to(device)    mix_fn = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, label_smoothing=label_smooth,                   num_classes=NUM_CLASSES, prob=1.0)    crit_train = SoftTargetCrossEntropy()    scaler = torch.amp.GradScaler("cuda" if device.type == "cuda" else "cpu")    best_f1, best_state, stage, epoch = -1.0, None, 1, 1    last_path = RESULTS_DIR / f"{name}_last.pth"    payload = None    if last_path.exists():        try:            payload = torch.load(last_path, map_location="cpu", weights_only=False)            model.load_state_dict(payload["model"])            best_f1 = payload["best_f1"]; best_state = payload["best_state"]            stage = payload["stage"]; epoch = payload["epoch"]+1            print(f"[RESUME] stage {stage} epoch {epoch}")        except: payload = None    while stage <= 2:        max_ep = 10 if stage == 1 else 50        use_mix = stage == 2        if stage == 1:            for p in model.parameters(): p.requires_grad = False            head = getattr(model, "head", None) or getattr(model, "classifier", None)            if head is None:                lins = [m for m in model.modules() if isinstance(m, nn.Linear)]; head = lins[-1]            for p in head.parameters(): p.requires_grad = True            opt = optim.AdamW(head.parameters(), lr=1e-3, weight_decay=wd)            crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smooth)            print(f"  Stage 1: head probe")        else:            for p in model.parameters(): p.requires_grad = True            hk = ("head","classifier","fc")            hp = [p for n,p in model.named_parameters() if any(k in n for k in hk)]            bp = [p for n,p in model.named_parameters() if not any(k in n for k in hk)]            opt = optim.AdamW([{"params":hp,"lr":1e-4},{"params":bp,"lr":1e-5}], weight_decay=wd)            crit = crit_train; print("  Stage 2: full fine-tune")        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_ep)        pat = 0        while epoch <= max_ep:            t0 = time.time()            tl, ta = train_one_epoch(model, tr_loader, crit, opt, scaler,                                     mix_fn if use_mix else None, device)            v = evaluate(model, va_loader, device); sch.step()            imp = v["macro_f1"] > best_f1            if imp: best_f1 = float(v["macro_f1"]); best_state = cpu_sd(model.state_dict()); pat = 0            else: pat += 1            save_ckpt(last_path, {"stage":stage,"epoch":epoch,"model":cpu_sd(model.state_dict()),                                   "opt":opt.state_dict(),"sch":sch.state_dict(),"scaler":scaler.state_dict(),                                   "best_f1":best_f1,"best_state":best_state})            mark = " *" if imp else ""            print(f"    Ep {epoch:02d}/{max_ep} | TrL {tl:.4f} TrA {ta:.4f} | "                  f"VaL {v['loss']:.4f} VaA {v['accuracy']:.4f} F1 {v['macro_f1']:.4f}{mark} | {time.time()-t0:.0f}s", flush=True)            if pat >= patience: print(f"    Early stop stage {stage}"); break            epoch += 1        stage += 1; epoch = 1    if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()})    save_ckpt(ckpt_path, {"model":name,"timm_name":timm_name,"state_dict":cpu_sd(model.state_dict()),                           "img_size":img_size,"val_f1":best_f1})    if last_path.exists(): last_path.unlink()    print(f"  Saved: {ckpt_path} (val_f1={best_f1:.4f})")    del model; torch.cuda.empty_cache()    return torch.load(ckpt_path, map_location="cpu", weights_only=False)print("Training loop ready.")

---## Phase 2b: Train All 4 Models

In [ ]:
# ============================================================# CELL 12: Train/Load All 4 Models# ============================================================trained_models = {}for cfg in MODELS_CONFIG:    ckpt_path = RESULTS_DIR / f"{cfg['name']}_best.pth"    if ckpt_path.exists():        print(f"[SKIP] {cfg['name']}"); trained_models[cfg["name"]] = torch.load(ckpt_path, map_location="cpu", weights_only=False)    else:        extra = {"strong": True}        if cfg["name"] == "convnext_tiny_v2":            extra.update({"label_smooth": 0.1, "wd": 0.05, "dp": 0.1})        ckpt = train_model(cfg["name"], cfg["timm_name"], cfg["img_size"], cfg["batch_size"],                           X_train, y_train, X_val, y_val, device, **extra)        trained_models[cfg["name"]] = ckptprint(f"\nAll models ready: {list(trained_models.keys())}")

In [ ]:
# ============================================================# CELL 13: Evaluation Helper# ============================================================def eval_on_dataset(model_name, ckpt, paths, labels, dataset_name):    model = build_model(ckpt["timm_name"])    model.load_state_dict(ckpt["state_dict"]); model.eval().to(device)    tf = get_transforms(ckpt["img_size"], False)    ds = PotatoDataset(paths, labels, tf)    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)    r = evaluate(model, loader, device)    r["model_name"] = model_name; r["dataset"] = dataset_name    ba = balanced_accuracy_score(labels, r["predictions"])    print(f"\n--- {model_name} on {dataset_name} (n={len(labels)}) ---")    print(f"  Acc={r['accuracy']:.4f}  Macro-F1={r['macro_f1']:.4f}  Balanced-Acc={ba:.4f}")    print(classification_report(labels, r["predictions"], target_names=IPD_CLASS_NAMES, digits=4, zero_division=0))    del model; torch.cuda.empty_cache()    return r

---## Phase 3: Cross-Domain Evaluation

In [ ]:
# ============================================================# CELL 14: IPD Test Evaluation# ============================================================print("=" * 60); print("IPD TEST SET"); print("=" * 60)ipd_results = {}for name in trained_models:    ipd_results[name] = eval_on_dataset(name, trained_models[name], X_test, y_test, "IPD Test")print("\n--- IPD TEST SUMMARY ---")print(f"{'Model':25s} {'Acc':>8s} {'F1':>8s}")for n, r in ipd_results.items():    print(f"{n:25s} {r['accuracy']:8.4f} {r['macro_f1']:8.4f}")

In [ ]:
# ============================================================# CELL 15: PLD Cross-Dataset Evaluation# ============================================================print("=" * 60); print("PLD CROSS-DATASET"); print("=" * 60)pld_results = {}for name in trained_models:    full = eval_on_dataset(name, trained_models[name], pld_paths, pld_labels.tolist(), "PLD Full")    clean_paths = [p for p, m in zip(pld_paths, clean_mask) if m]    clean_y = pld_labels[clean_mask].tolist()    clean = eval_on_dataset(name, trained_models[name], clean_paths, clean_y, "PLD Clean")    pld_results[name] = {"full": full, "clean": clean}print("\n--- CROSS-DOMAIN COMPARISON ---")print(f"{'Model':25s} {'IPD-F1':>8s} {'PLD-Full':>10s} {'PLD-Clean':>10s} {'Gap':>8s}")for n in trained_models:    f1_ipd = ipd_results[n]["macro_f1"]    f1_full = pld_results[n]["full"]["macro_f1"]    f1_clean = pld_results[n]["clean"]["macro_f1"]    print(f"{n:25s} {f1_ipd:8.4f} {f1_full:10.4f} {f1_clean:10.4f} {f1_ipd-f1_full:8.4f}")json.dump({n: {"ipd_f1": ipd_results[n]["macro_f1"], "pld_full_f1": pld_results[n]["full"]["macro_f1"],               "pld_clean_f1": pld_results[n]["clean"]["macro_f1"]}           for n in trained_models}, open(RESULTS_DIR/"cross_domain.json","w"), indent=2)

---## Phase 3b: Error Analysis

In [ ]:
# ============================================================# CELL 16: PLD Error Analysis# ============================================================best_pld = max(pld_results.keys(), key=lambda n: pld_results[n]["full"]["macro_f1"])print(f"Best PLD model: {best_pld}")model = build_model(trained_models[best_pld]["timm_name"])model.load_state_dict(trained_models[best_pld]["state_dict"]); model.eval().to(device)tf = get_transforms(trained_models[best_pld]["img_size"], False)ds = PotatoDataset(pld_paths, pld_labels.tolist(), tf)loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)all_probs = []with torch.no_grad():    for x, _ in loader:        x = x.to(device)        with torch.amp.autocast(device.type if device.type == "cuda" else "cpu"):            out = model(x)        all_probs.append(torch.softmax(out.float(), dim=1).cpu().numpy())probs = np.concatenate(all_probs); preds = probs.argmax(1); max_conf = probs.max(1)records = []for i in range(len(pld_paths)):    records.append({"path": pld_paths[i], "group": pld_groups[i],                     "true": IPD_CLASS_NAMES[pld_labels[i]], "pred": IPD_CLASS_NAMES[preds[i]],                     "correct": preds[i] == pld_labels[i], "confidence": float(max_conf[i])})err_df = pd.DataFrame(records)n_correct = err_df["correct"].sum(); n_total = len(err_df)print(f"\nCorrect: {n_correct}/{n_total} ({n_correct/n_total*100:.1f}%)")errors = err_df[~err_df["correct"]]print(f"High-conf errors (>0.8): {(errors['confidence']>0.8).sum()}")print(f"\nBy PLD group:")for g in sorted(err_df["group"].unique()):    grp = err_df[err_df["group"]==g]; e = (~grp["correct"]).sum()    print(f"  {g:20s}: {e}/{len(grp)} ({e/len(grp)*100:.1f}%)")err_df.to_csv(RESULTS_DIR/"pld_error_analysis.csv", index=False)del model; torch.cuda.empty_cache()

In [ ]:
# ============================================================# CELL 17: Error Visualization Grids# ============================================================def show_grid(df, title, n=12):    subset = df.head(n)    if len(subset) == 0: print(f"No images: {title}"); return    cols = min(4, len(subset)); rows = (len(subset) + cols - 1) // cols    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))    axes = np.array(axes).flatten()    for i, (_, row) in enumerate(subset.iterrows()):        try:            img = Image.open(row["path"]).convert("RGB")            axes[i].imshow(img)            c = "green" if row["correct"] else "red"            axes[i].set_title(f"T:{row['true']}\\nP:{row['pred']}\\nC:{row['confidence']:.2f}", color=c, fontsize=9)        except: axes[i].text(0.5,0.5,"?", ha="center", va="center")        axes[i].axis("off")    for j in range(i+1, len(axes)): axes[j].axis("off")    plt.suptitle(title, fontsize=13, fontweight="bold"); plt.tight_layout()    plt.savefig(RESULTS_DIR / f"grid_{title.replace(' ','_').lower()}.png", dpi=120, bbox_inches="tight")    plt.show()errors = err_df[~err_df["correct"]]for cls in IPD_CLASS_NAMES:    show_grid(errors[errors["true"]==cls].sort_values("confidence", ascending=False),              f"False {cls}")show_grid(errors.sort_values("confidence", ascending=False), "High-Conf Mistakes")show_grid(err_df[err_df["correct"]].sort_values("confidence"), "Low-Conf Correct")

---## Phase 3c: Grad-CAM

In [ ]:
# ============================================================# CELL 18: Grad-CAM# ============================================================try:    from pytorch_grad_crl import GradCAM    from pytorch_grad_crl.utils import show_cam_on_image    HAS_GC = Trueexcept ImportError:    try:        from grad_cam import GradCAM        from grad_cam.utils.image import show_cam_on_image        HAS_GC = True    except ImportError: HAS_GC = Falseif HAS_GC:    def get_target_layer(model, name):        if "convnext" in name: return model.features[-1]        if "efficientnet" in name: return model.features[-1]        if "swin" in name: return model.features[-1]        for m in reversed(list(model.modules())):            if isinstance(m, nn.Conv2d): return m        return None    def show_gradcam_image(model_name, ckpt, img_path, true_label=None):        model = build_model(ckpt["timm_name"]); model.load_state_dict(ckpt["state_dict"])        model.eval().to(device)        tl = get_target_layer(model, ckpt["timm_name"])        if tl is None: print("No target layer"); return        cam = GradCAM(model=model, target_layers=[tl], use_cuda=device.type=="cuda")        img = Image.open(img_path).convert("RGB")        tf = get_transforms(ckpt["img_size"], False)        x = tf(img).unsqueeze(0).to(device)        with torch.no_grad():            out = model(x); probs = torch.softmax(out, dim=1)            pred = out.argmax(1).item(); conf = probs[0, pred].item()        gcam = cam(input_tensor=x)[0]        vis = show_cam_on_image(np.array(img.resize((ckpt["img_size"], ckpt["img_size"])))/255.0, gcam, use_rgb=True)        fig, axes = plt.subplots(1, 3, figsize=(15, 5))        axes[0].imshow(img); axes[0].set_title(f"Original\\nTrue: {IPD_CLASS_NAMES[true_label] if true_label is not None else '?'}")        axes[1].imshow(gcam, cmap="jet"); axes[1].set_title("Grad-CAM")        axes[2].imshow(vis); axes[2].set_title(f"Overlay\\nPred: {IPD_CLASS_NAMES[pred]} ({conf:.2f})")        for a in axes: a.axis("off")        plt.suptitle(f"{model_name} - {Path(img_path).name}", fontweight="bold")        plt.tight_layout(); plt.savefig(RESULTS_DIR/f"gradcam_{Path(img_path).stem}.png", dpi=120); plt.show()        del model; torch.cuda.empty_cache()    best_m = max(pld_results.keys(), key=lambda n: pld_results[n]["full"]["macro_f1"])    for idx in np.random.choice(len(pld_paths), min(4, len(pld_paths)), replace=False):        show_gradcam_image(best_m, trained_models[best_m], pld_paths[idx], pld_labels[idx])else:    print("Grad-CAM not available. pip install grad-cam")

In [ ]:
# ============================================================# CELL 19: Confusion Matrices# ============================================================best_n = max(ipd_results.keys(), key=lambda n: ipd_results[n]["macro_f1"])fig, axes = plt.subplots(1, 3, figsize=(18, 5))cm1 = confusion_matrix(y_test, ipd_results[best_n]["predictions"])sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues", ax=axes[0],            xticklabels=IPD_CLASS_NAMES, yticklabels=IPD_CLASS_NAMES)axes[0].set_title(f"IPD Test ({best_n})"); axes[0].set_ylabel("True"); axes[0].set_xlabel("Pred")cm2 = confusion_matrix(pld_labels, pld_results[best_n]["full"]["predictions"])sns.heatmap(cm2, annot=True, fmt="d", cmap="Oranges", ax=axes[1],            xticklabels=IPD_CLASS_NAMES, yticklabels=IPD_CLASS_NAMES)axes[1].set_title(f"PLD Full ({best_n})"); axes[1].set_ylabel("True"); axes[1].set_xlabel("Pred")cm2n = cm2.astype(float)/cm2.sum(axis=1, keepdims=True)sns.heatmap(cm2n, annot=True, fmt=".2f", cmap="Oranges", ax=axes[2],            xticklabels=IPD_CLASS_NAMES, yticklabels=IPD_CLASS_NAMES)axes[2].set_title(f"PLD Normalized"); axes[2].set_ylabel("True"); axes[2].set_xlabel("Pred")plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")plt.tight_layout(); plt.savefig(RESULTS_DIR/"confusion_matrices.png", dpi=150); plt.show()

---## Phase 4: Production Robustness

In [ ]:
# ============================================================# CELL 20: Robustness Benchmark# ============================================================import cv2def corrupt(img_path, ctype, sev=1):    img = cv2.imread(str(img_path))    if img is None: return None    if ctype == "blur":        return cv2.GaussianBlur(img, (3+sev*2, 3+sev*2), 0)    elif ctype == "noise":        n = np.random.normal(0, sev*10, img.shape).astype(np.float32)        return np.clip(img.astype(np.float32)+n, 0, 255).astype(np.uint8)    elif ctype == "jpeg":        _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, max(10, 90-sev*15)])        return cv2.imdecode(enc, 1)    elif ctype == "brightness":        return np.clip(img.astype(np.float32)*(1+(sev-2)*0.3), 0, 255).astype(np.uint8)    elif ctype == "rotation":        h,w = img.shape[:2]        M = cv2.getRotationMatrix2D((w/2,h/2), sev*5, 1.0)        return cv2.warpAffine(img, M, (w,h))    return imgprint("Robustness benchmark...")sample_idx = np.random.choice(len(X_test), min(100, len(X_test)), replace=False)sp = [X_test[i] for i in sample_idx]; sl = [y_test[i] for i in sample_idx]robust_rows = []for ctype in ["blur", "noise", "jpeg", "brightness", "rotation"]:    for sev in [1, 2, 3]:        tmp_dir = RESULTS_DIR/"tmp_cr"; tmp_dir.mkdir(exist_ok=True)        cpaths = []        for p in sp:            img = corrupt(p, ctype, sev)            tp = tmp_dir/f"{Path(p).stem}_{ctype}_s{sev}.png"            if img is not None: cv2.imwrite(str(tp), img); cpaths.append(str(tp))            else: cpaths.append(p)        ds = PotatoDataset(cpaths, sl, get_transforms(trained_models[best_n]["img_size"], False))        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)        r = evaluate(model, loader, device) if 'model' in dir() else {"accuracy":0,"macro_f1":0}        robust_rows.append({"corruption":ctype, "severity":sev, "accuracy":r["accuracy"], "f1":r["macro_f1"]})        print(f"  {ctype:12s} s={sev}: F1={r['macro_f1']:.4f}")        for cp in cpaths:            if "tmp_cr" in cp:                try: os.remove(cp); pass                except: passrobust_df = pd.DataFrame(robust_rows)robust_df.to_csv(RESULTS_DIR/"robustness.csv", index=False)fig, ax = plt.subplots(figsize=(10, 5))for c in robust_df["corruption"].unique():    d = robust_df[robust_df["corruption"]==c]    ax.plot(d["severity"], d["f1"], marker="o", label=c)ax.set_xlabel("Severity"); ax.set_ylabel("Macro-F1")ax.set_title("Robustness Under Corruptions"); ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.savefig(RESULTS_DIR/"robustness.png", dpi=150); plt.show()

In [ ]:
# ============================================================# CELL 21: Out-of-Distribution Detection# ============================================================# Generate OOD imagesood_paths = []for i in range(30):    noise = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)    tp = RESULTS_DIR/f"ood_{i}.png"; cv2.imwrite(str(tp), noise); ood_paths.append(str(tp))model = build_model(trained_models[best_n]["timm_name"])model.load_state_dict(trained_models[best_n]["state_dict"]); model.eval().to(device)tf = get_transforms(trained_models[best_n]["img_size"], False)def get_probs(paths):    ds = PotatoDataset(paths, [0]*len(paths), tf)    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)    all_p = []    with torch.no_grad():        for x, _ in loader:            x = x.to(device)            with torch.amp.autocast(device.type if device.type == "cuda" else "cpu"):                out = model(x)            all_p.append(torch.softmax(out.float(), dim=1).cpu().numpy())    return np.concatenate(all_p)known_p = get_probs(X_test[:200]).max(axis=1)ood_p = get_probs(ood_paths).max(axis=1)threshold = np.percentile(known_p, 5)print(f"OOD Detection:")print(f"  Known max_prob: mean={known_p.mean():.3f}")print(f"  OOD max_prob:   mean={ood_p.mean():.3f}")print(f"  Threshold:      {threshold:.3f}")print(f"  Known reject:   {(known_p < threshold).mean()*100:.1f}%")print(f"  OOD accept:     {(ood_p >= threshold).mean()*100:.1f}%")fig, ax = plt.subplots(figsize=(8, 4))ax.hist(known_p, bins=20, alpha=0.7, label="Known", density=True)ax.hist(ood_p, bins=20, alpha=0.7, label="OOD", density=True)ax.axvline(threshold, color="red", ls="--", label=f"T={threshold:.3f}")ax.set_xlabel("Max Prob"); ax.set_title("OOD Detection"); ax.legend()plt.tight_layout(); plt.savefig(RESULTS_DIR/"ood.png", dpi=150); plt.show()del model; torch.cuda.empty_cache()for p in ood_paths: os.remove(p)

In [ ]:
# ============================================================# CELL 22: Confidence Calibration (Temperature Scaling)# ============================================================from scipy.optimize import minimize_scalardef calibrate(model_name, ckpt, val_paths, val_labels, test_paths, test_labels):    model = build_model(ckpt["timm_name"]); model.load_state_dict(ckpt["state_dict"])    model.eval().to(device); tf = get_transforms(ckpt["img_size"], False)    def get_logits(paths, labels):        ds = PotatoDataset(paths, labels, tf)        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)        logits, labs = [], []        with torch.no_grad():            for x, y in loader:                x = x.to(device)                with torch.amp.autocast(device.type if device.type == "cuda" else "cpu"):                    out = model(x)                logits.append(out.float().cpu().numpy()); labs.append(y.numpy())        return np.concatenate(logits), np.concatenate(labs)    vl, vy = get_logits(val_paths[:1000], val_labels[:1000])    tl, ty = get_logits(test_paths, test_labels)    # Fit temperature    def nll(T):        s = np.exp(vl/T) / np.exp(vl/T).sum(axis=1, keepdims=True)        return -np.log(s[np.arange(len(vy)), vy]+1e-10).mean()    T = minimize_scalar(nll, bounds=(0.1,10), method="bounded").x    print(f"Temperature: {T:.3f}")    def ece(probs, labels, n=15):        conf = probs.max(1); pred = probs.argmax(1); acc = (pred==labels).astype(float)        bins = np.linspace(0,1,n+1); e = 0        for i in range(n):            m = (conf>bins[i])&(conf<=bins[i+1])            if m.sum()>0: e += m.sum()/len(labels)*abs(acc[m].mean()-conf[m].mean())        return e    def brier(probs, labels):        return ((probs - np.eye(NUM_CLASSES)[labels])**2).sum(1).mean()    # Before    p_raw = np.exp(tl)/np.exp(tl).sum(1, keepdims=True)    # After    p_scaled = np.exp(tl/T)/np.exp(tl/T).sum(1, keepdims=True)    print(f"\n{'Metric':15s} {'Before':>10s} {'After':>10s}")    print(f"{'Accuracy':15s} {(p_raw.argmax(1)==ty).mean():10.4f} {(p_scaled.argmax(1)==ty).mean():10.4f}")    print(f"{'ECE':15s} {ece(p_raw,ty):10.4f} {ece(p_scaled,ty):10.4f}")    print(f"{'Brier':15s} {brier(p_raw,ty):10.4f} {brier(p_scaled,ty):10.4f}")    # Reliability diagram    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    for ax, p, t in [(axes[0], p_raw, "Before"), (axes[1], p_scaled, "After")]:        conf = p.max(1); pred = p.argmax(1); acc = (pred==ty).astype(float)        bins = np.linspace(0,1,16); bc, ba, bconf = [], [], []        for i in range(16):            m = (conf>bins[i])&(conf<=bins[i+1])            if m.sum()>0: bc.append((bins[i]+bins[i+1])/2); ba.append(acc[m].mean()); bconf.append(conf[m].mean())        ax.plot([0,1],[0,1],"k--"); ax.plot(bconf, ba, "s-")        ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_title(t); ax.grid(True, alpha=0.3)    plt.suptitle(f"Calibration: {model_name}", fontweight="bold")    plt.tight_layout(); plt.savefig(RESULTS_DIR/"calibration.png", dpi=150); plt.show()    del model; torch.cuda.empty_cache()    return TT = calibrate(best_n, trained_models[best_n], X_val, y_val, X_test, y_test)

---## Phase 4b: Statistical Validation

In [ ]:
# ============================================================# CELL 23: Bootstrap 95% Confidence Intervals# ============================================================def boot_ci(yt, yp, fn, n=1000):    scores = [fn(yt[np.random.choice(len(yt), len(yt), replace=True)],                  yp[np.random.choice(len(yp), len(yp), replace=True)]) for _ in range(n)]    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)print("Bootstrap 95% CI (1000 reps)")print(f"{'Dataset':15s} {'Metric':10s} {'Score':>8s} {'Lower':>8s} {'Upper':>8s}")print("-" * 55)for dname, yt, yp in [("IPD Test", np.array(y_test), ipd_results[best_n]["predictions"]),                        ("PLD Full", pld_labels, pld_results[best_n]["full"]["predictions"])]:    acc, alo, ahi = boot_ci(yt, yp, lambda a,b: (a==b).mean())    f1, flo, fhi = boot_ci(yt, yp, lambda a,b: f1_score(a,b,average="macro",zero_division=0))    print(f"{dname:15s} {'Accuracy':10s} {acc:8.4f} {alo:8.4f} {ahi:8.4f}")    print(f"{'':15s} {'Macro-F1':10s} {f1:8.4f} {flo:8.4f} {fhi:8.4f}")

---## Phase 5: Final Decision

In [ ]:
# ============================================================# CELL 24: ConvNeXt v1 vs v2 Comparison# ============================================================v1n, v2n = "convnext_tiny_v1", "convnext_tiny_v2"if v1n in trained_models and v2n in trained_models:    print("v1 vs v2 Comparison")    print(f"\n{'Metric':15s} {'v1':>10s} {'v2':>10s}")    print("-" * 37)    for d in ["ipd_results", "pld_results"]:        if d == "ipd_results":            v1, v2 = ipd_results[v1n]["macro_f1"], ipd_results[v2n]["macro_f1"]            print(f"{'IPD F1':15s} {v1:10.4f} {v2:10.4f}")        else:            v1, v2 = pld_results[v1n]["full"]["macro_f1"], pld_results[v2n]["full"]["macro_f1"]            print(f"{'PLD F1':15s} {v1:10.4f} {v2:10.4f}")else:    print("Need both v1 and v2 for comparison")

In [ ]:
# ============================================================# CELL 25: Production Inference# ============================================================import ioclass PotatoClassifier:    def __init__(self, model_path, threshold=0.5):        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)        self.model = build_model(ckpt["timm_name"])        self.model.load_state_dict(ckpt["state_dict"]); self.model.eval()        self.img_size = ckpt["img_size"]; self.threshold = threshold        self.class_names = IPD_CLASS_NAMES        self.tf = transforms.Compose([            transforms.Resize(int(self.img_size*1.14)), transforms.CenterCrop(self.img_size),            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")        self.model.to(self.device)    @torch.no_grad()    def predict(self, img_input):        if isinstance(img_input, (str, Path)): img = Image.open(img_input).convert("RGB")        elif isinstance(img_input, bytes): img = Image.open(io.BytesIO(img_input)).convert("RGB")        elif isinstance(img_input, Image.Image): img = img_input.convert("RGB")        else: raise ValueError(f"Bad input: {type(img_input)}")        x = self.tf(img).unsqueeze(0).to(self.device)        with torch.amp.autocast(self.device.type if self.device.type == "cuda" else "cpu"):            probs = torch.softmax(self.model(x).float(), dim=1)[0].cpu().numpy()        pred = probs.argmax(); conf = float(probs[pred])        return {"prediction": self.class_names[pred] if conf >= self.threshold else "Unknown",                "confidence": round(conf, 4),                "probabilities": {n: round(float(probs[i]),4) for i,n in enumerate(self.class_names)},                "status": "accepted" if conf >= self.threshold else "rejected"}# Demobest_path = RESULTS_DIR / f"{best_n}_best.pth"clf = PotatoClassifier(best_path)print(f"Classifier loaded: {best_n}")# Save production modeltorch.save({"model_name": best_n, "state_dict": clf.model.state_dict(),            "timm_name": trained_models[best_n]["timm_name"], "img_size": clf.img_size,            "class_names": clf.class_names, "threshold": clf.threshold},           RESULTS_DIR/"production_model.pth")print(f"Production model saved: {RESULTS_DIR/'production_model.pth'}")

---## Production Gate

In [ ]:
# ============================================================# CELL 26: Production Gate# ============================================================ipd_f1 = ipd_results[best_n]["macro_f1"]pld_f1 = pld_results[best_n]["full"]["macro_f1"]gap = ipd_f1 - pld_f1def status(metric, green, yellow):    if metric >= green: return "GREEN"    if metric >= yellow: return "YELLOW"    return "RED"print("=" * 70)print("PRODUCTION READINESS REPORT")print("=" * 70)rows = [    ("IPD Test Acc", ipd_results[best_n]["accuracy"], status(ipd_results[best_n]["accuracy"], 0.99, 0.95)),    ("IPD Test F1", ipd_f1, status(ipd_f1, 0.99, 0.95)),    ("PLD Full F1", pld_f1, status(pld_f1, 0.70, 0.50)),    ("PLD Clean F1", pld_results[best_n]["clean"]["macro_f1"], status(pld_results[best_n]["clean"]["macro_f1"], 0.80, 0.60)),    ("Domain Gap", gap, status(1-gap, 0.90, 0.70)),]print()print(f"{'Metric':20s} {'Value':>10s} {'Status':>10s}")print("-" * 42)for name, val, st in rows:    print(f"{name:20s} {val:10.4f} {st:>10s}")statuses = [r[2] for r in rows]if "RED" in statuses:    overall = "RED"; verdict = "NOT PRODUCTION READY"elif "YELLOW" in statuses:    overall = "YELLOW"; verdict = "NEEDS MORE VALIDATION"else:    overall = "GREEN"; verdict = "PRODUCTION READY"print()print("=" * 70)print("OVERALL:", overall)print("VERDICT:", verdict)print("=" * 70)print()print("KEY FINDINGS:")findings = "EXCELLENT" if ipd_f1 > 0.99 else "GOOD" if ipd_f1 > 0.95 else "NEEDS WORK"print("  1. IPD (in-distribution): " + str(round(ipd_f1, 4)) + " F1 - " + findings)pld_status = "GOOD" if pld_f1 > 0.6 else "NEEDS IMPROVEMENT"print("  2. PLD (unseen field): " + str(round(pld_f1, 4)) + " F1 - " + pld_status)gap_status = "MANAGEABLE" if gap < 0.3 else "SIGNIFICANT"print("  3. Domain gap: " + str(round(gap, 4)) + " - " + gap_status)print()print("RECOMMENDATIONS:")if pld_f1 > 0.5:    print("  - Model shows reasonable cross-domain generalization.")else:    print("  - Train with field-condition data for better generalization.")print("  - Monitor confidence in production for OOD detection.")print("  - Collect real smartphone images for further validation.")print("  - Consider domain adaptation if deploying to field conditions.")report = {"model": best_n, "ipd_f1": ipd_f1, "pld_f1": pld_f1,          "gap": gap, "overall": overall, "verdict": verdict}json.dump(report, open(RESULTS_DIR/"production_gate.json","w"), indent=2)print()print("Report saved:", RESULTS_DIR/"production_gate.json")print("All results in:", RESULTS_DIR)